# Fine-tune Llama 3.2 for Price Prediction (Tope AI Labs)

**Week 7 – Community contribution:** Fine-tune **Meta Llama 3.2 3B** on a **small dataset** from Week 7 samples using **Hugging Face** (transformers, datasets, peft, trl) and **QLoRA** to predict product prices from descriptions.

- **Model:** `meta-llama/Llama-3.2-3B`
- **Task:** Given "What does this cost to the nearest dollar?" + product summary → predict "Price is $X.00"
- **Data:** Week 7–style prompts from Hugging Face (small subset for quick runs)
- **Hardware:** Runs on a single GPU (e.g. Colab T4) with 4-bit QLoRA

## 1. Install dependencies

If you see **"No module named pip"**, run the cell below first (it installs pip into your venv). Then re-run the *next* cell to install packages. Restart the kernel after installing.

In [2]:
# Only run this if your venv has no pip ("No module named pip"). Restart kernel after.
import sys
import subprocess
subprocess.run([sys.executable, "-m", "ensurepip", "--upgrade"], check=True)
print("pip installed. Restart the kernel, then run the cell below to install packages.")

pip installed. Restart the kernel, then run the cell below to install packages.


In [1]:
# Run locally or in Colab to install deps (skip if already in your env)
%pip install -q transformers datasets peft trl bitsandbytes accelerate

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Imports and config

In [3]:
import os
from datetime import datetime
from getpass import getpass

import torch
from datasets import load_dataset
from huggingface_hub import login
from peft import LoraConfig
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)
from trl import SFTConfig, SFTTrainer

In [4]:
# ── Model & data ─────────────────────────────────────────────
BASE_MODEL = "meta-llama/Llama-3.2-1B"
HF_USER = "tupskey"  # set your Hugging Face username

# Week 7 dataset on Hugging Face (prompt + completion format)
DATASET_NAME = "ed-donner/items_prompts_lite"

# Small dataset: use only this many training samples (for quick runs)
SMALL_TRAIN_SIZE = 2000
VAL_SIZE = 200

# ── QLoRA ────────────────────────────────────────────────────
QUANT_4_BIT = True
LORA_R = 32
LORA_ALPHA = 64
LORA_DROPOUT = 0.1
TARGET_MODULES = ["q_proj", "v_proj", "k_proj", "o_proj"]

# ── Training ────────────────────────────────────────────────
MAX_SEQUENCE_LENGTH = 128
EPOCHS = 1
BATCH_SIZE = 8
GRADIENT_ACCUMULATION_STEPS = 2
LEARNING_RATE = 2e-4
WARMUP_RATIO = 0.03
LR_SCHEDULER_TYPE = "cosine"
SAVE_STEPS = 100
LOG_STEPS = 10

RUN_NAME = f"price-{datetime.now():%Y-%m-%d_%H.%M.%S}-small"
OUTPUT_DIR = RUN_NAME
HUB_MODEL_ID = f"{HF_USER}/llama32-price-small"

# bf16 needs GPU with capability >= 8 (A100, etc.); safe default when no CUDA
try:
    capability = torch.cuda.get_device_capability()
    use_bf16 = capability[0] >= 8
except (AssertionError, RuntimeError):
    use_bf16 = False
    print("No CUDA GPU detected. use_bf16=False (fp16 will be used if you run on GPU later).")
    print("Training this model requires a GPU (e.g. Colab with T4/A100).")

No CUDA GPU detected. use_bf16=False (fp16 will be used if you run on GPU later).
Training this model requires a GPU (e.g. Colab with T4/A100).


## 3. Log in to Hugging Face

In [ ]:
# Use token from env or prompt
hf_token = os.environ.get("HF_TOKEN") or getpass("Hugging Face token: ")
login(token=hf_token, add_to_git_credential=True)

## 4. Load and prepare dataset (small subset)

In [ ]:
dataset = load_dataset(DATASET_NAME)

# Take a small subset for fast fine-tuning
train_raw = dataset["train"].select(range(min(SMALL_TRAIN_SIZE, len(dataset["train"]))))
val_raw = dataset["val"].select(range(min(VAL_SIZE, len(dataset["val"]))))

print(f"Train size: {len(train_raw)}, Val size: {len(val_raw)}")

In [ ]:
# SFTTrainer expects a single text field: concatenate prompt + completion
def add_text(example):
    example["text"] = example["prompt"] + example["completion"]
    return example

train_ds = train_raw.map(add_text, remove_columns=["prompt", "completion"])
val_ds = val_raw.map(add_text, remove_columns=["prompt", "completion"])

print("Sample:", train_ds[0]["text"][:200], "...")

## 5. Load tokenizer and model (4-bit QLoRA)

In [ ]:
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16 if use_bf16 else torch.float16,
    bnb_4bit_quant_type="nf4",
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=quant_config,
    device_map="auto",
)
model.generation_config.pad_token_id = tokenizer.pad_token_id

print(f"Model memory footprint: {model.get_memory_footprint() / 1e6:.1f} MB")

## 6. LoRA config and training args

In [14]:
lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=TARGET_MODULES,
)

training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=LEARNING_RATE,
    warmup_ratio=WARMUP_RATIO,
    lr_scheduler_type=LR_SCHEDULER_TYPE,
    fp16=not use_bf16,
    bf16=use_bf16,
    max_grad_norm=0.3,
    max_length=MAX_SEQUENCE_LENGTH,
    save_steps=SAVE_STEPS,
    logging_steps=LOG_STEPS,
    save_total_limit=2,
    optim="paged_adamw_32bit",
    report_to="none",
    run_name=RUN_NAME,
    eval_strategy="steps",
    eval_steps=SAVE_STEPS,
    dataset_text_field="text",
)


## 7. Train with SFTTrainer

In [ ]:
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    peft_config=lora_config,
    processing_class=tokenizer,
)

trainer.train()

## 8. Save and optionally push to Hub

In [ ]:
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

# Optional: push adapter to Hugging Face Hub
# trainer.model.push_to_hub(HUB_MODEL_ID, private=True)
# tokenizer.push_to_hub(HUB_MODEL_ID)
print(f"Saved to {OUTPUT_DIR}")

## 9. Quick inference example

In [ ]:
from peft import PeftModel

# Load base model + adapter from output dir (or from Hub)
base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=quant_config,
    device_map="auto",
)
model_inference = PeftModel.from_pretrained(base, OUTPUT_DIR)

prompt = "What does this cost to the nearest dollar?\n\nBrand new wireless Bluetooth headphones with noise canceling.\n\nPrice is $"
inputs = tokenizer(prompt, return_tensors="pt").to(model_inference.device)
out = model_inference.generate(**inputs, max_new_tokens=8, do_sample=False)
response = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
print("Model output:", response)